In [1]:
!pip -q install numpy scikit-learn sentence-transformers
!python --version
!pip freeze > requirements.txt

Python 3.13.18


In [2]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or not all(rec["text"].strip() for rec in records):
            raise ValueError("Provide nonempty records")
        self.records = records
        self.backend = backend
        corpus = [rec["text"] for rec in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(corpus)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(MODEL_NAME, device="cpu")
            self.check_length(corpus)
            self.matrix = self.model.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")

    def check_length(self, texts):
        limit = self.model.max_seq_length
        for item in texts:
            if len(self.model.tokenizer(item, truncation=False)["input_ids"]) > limit:
                raise ValueError("Input exceeds encoder token limit")

    def _scores(self, query):
        if self.backend == "tfidf":
            vec = self.vectorizer.transform([query])
            if vec.nnz == 0:
                return None
            return (vec @ self.matrix.T).toarray()[0]
        self.check_length([query])
        vec = self.model.encode([query], normalize_embeddings=True)[0]
        return self.matrix @ vec

    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        scores = self._scores(query)
        if scores is None:
            return []
        wanted = filters or {}
        pool = [i for i, rec in enumerate(self.records)
                if all(rec.get(key) == val for key, val in wanted.items())]
        pool.sort(key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in pool[:k]]


def show_hits(hits):
    if not hits:
        print("No matching records.")
    for rec, score in hits:
        print(rec["id"], f"score={score:.3f}")
        print(rec["text"])

Writing lab6_search.py


In [3]:
%%writefile task2_chunks.py
import json
import re
from pathlib import Path


def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]


def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    step = size - overlap
    out = []
    for first in range(0, len(spans), step):
        last = min(first + size, len(spans))
        out.append((offset + spans[first][0], offset + spans[last - 1][1]))
        if last == len(spans):
            break
    return out


def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    bounds = [0, *(m.end() for m in re.finditer(r"[.!?]\s+", text)), len(text)]
    out = []
    current = None
    used = 0
    for lo, hi in zip(bounds, bounds[1:]):
        count = len(text[lo:hi].split())
        if count == 0:
            continue
        if count > size:
            if current:
                out.append(current)
            out.extend(windows(text[lo:hi], size, 0, lo))
            current, used = None, 0
        elif current and used + count <= size:
            current = (current[0], hi)
            used += count
        else:
            if current:
                out.append(current)
            current, used = (lo, hi), count
    if current:
        out.append(current)
    return out


def make_chunks(docs, strategy):
    rows = []
    for doc_id, text in docs.items():
        if strategy == "sentence":
            spans = sentence_chunks(text)
        else:
            spans = windows(text, 40, int(strategy))
        for n, (lo, hi) in enumerate(spans):
            rows.append({"id": f"{doc_id}:{n}", "doc": doc_id,
                         "start": lo, "end": hi, "text": text[lo:hi]})
    return rows


RULE_PAIRS = {
    "late": ("Late submission is normally penalized.",
             "Illness extensions require medical evidence before the deadline."),
    "password": ("Reset a forgotten password using the recovery link.",
                 "Never share recovery codes."),
    "assignment": ("Upload assignments through the learning portal.",
                   "Submit the final file as a PDF."),
    "contractor": ("Contractors can work remotely only if the signed contract permits it.",
                   "Employee rules do not apply to contractors."),
    "printer": ("For blank printer pages, check ink levels and run the cleaning utility.",
                "Print a test page afterwards."),
}


def compare_strategies(docs):
    base_total = sum(len(t.split()) for t in docs.values())
    print(f"{'strategy':10} {'chunks':>6} {'avg words':>9} {'max words':>9} "
          f"{'dup words':>9} {'rule+exception together':>24}")
    for strategy in ("0", "10", "sentence"):
        rows = make_chunks(docs, strategy)
        lengths = [len(r["text"].split()) for r in rows]
        kept, marks = 0, []
        for doc, (rule, exc) in RULE_PAIRS.items():
            if doc not in docs:
                continue
            together = any(rule in r["text"] and exc in r["text"]
                           for r in rows if r["doc"] == doc)
            kept += together
            marks.append(f"{doc}={'Y' if together else 'N'}")
        print(f"{strategy:10} {len(rows):>6} {sum(lengths) / len(lengths):>9.1f} "
              f"{max(lengths):>9} {sum(lengths) - base_total:>9} "
              f"{kept}/{len(marks)}  ({', '.join(marks)})")


def run_checks():
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    lengthy = " ".join(["word"] * 95) + "."
    pieces = sentence_chunks(lengthy)
    assert all(len(lengthy[a:b].split()) <= 40 for a, b in pieces)
    assert len(pieces) == 3
    assert sentence_chunks("") == []
    sample = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ("0", "10", "sentence"):
        for chunk in make_chunks(sample, strategy):
            assert chunk["text"] == sample[chunk["doc"]][chunk["start"]:chunk["end"]]
            print(strategy, chunk)
    print("All boundary tests passed.\n")

    abbreviation = "Dr. Saeed Anwar coordinates the office. Contact him first."
    print("Abbreviation demo:")
    for a, b in sentence_chunks(abbreviation, size=5):
        print("  ", repr(abbreviation[a:b]))

    path = Path("corpus.json")
    if path.exists():
        corpus = json.loads(path.read_text(encoding="utf-8"))
        print("\nCorpus words per document:",
              {name: len(body.split()) for name, body in corpus.items()})
        print()
        compare_strategies(corpus)


if __name__ == "__main__":
    run_checks()

Writing task2_chunks.py


In [4]:
%%writefile create_dataset.py
import json
import sys
from pathlib import Path

entries = [
    ("late",
     "This synthetic handbook section describes how late work is handled in "
     "the teaching programme. Late submission is normally penalized. The "
     "penalty is ten percent of the available marks for each calendar day "
     "after the deadline, and work that is more than five days late receives "
     "no marks. Dr. Saeed Anwar coordinates the assessment office and "
     "publishes the final penalty table every semester. Students who miss a "
     "deadline for personal reasons should contact the coordinator as early "
     "as possible. Illness extensions require medical evidence before the "
     "deadline. The evidence must be a signed note from a licensed doctor "
     "that states the dates of the illness. Extensions for non-medical "
     "reasons need written approval from the course coordinator and are "
     "granted only in rare cases. An older version of this handbook allowed "
     "students to send medical evidence up to seven days after the "
     "deadline, but that rule was withdrawn and must not be used. Group "
     "assignments follow the same rules, and one extension request covers "
     "the whole group when every member is affected. Requests sent after "
     "marking has started cannot be accepted. All decisions are recorded in "
     "the student portal so that the history of each request can be checked "
     "later.",
     "Illness extensions require medical evidence before the deadline."),
    ("password",
     "This synthetic IT help section explains account access for students "
     "and staff. Reset a forgotten password using the recovery link. The "
     "link is shown below the login box on the sign-in page and is sent to "
     "the registered university email address. It stays valid for thirty "
     "minutes, and a new request cancels any earlier link. Never share "
     "recovery codes. The help desk will never ask for a password or a "
     "recovery code by phone, chat or email. If the registered email "
     "address is no longer available, the user must visit the IT desk with "
     "a student card so that the identity can be checked in person. Staff "
     "accounts are reset by the IT desk only, because they give access to "
     "grades and personal records. After three wrong password attempts the "
     "account is locked for fifteen minutes to protect it from guessing. "
     "Passwords should contain at least twelve characters and should not be "
     "reused on other websites. Students are encouraged to enable the "
     "second step sign-in option in their profile settings, which adds a "
     "code from a phone application to every new login.",
     "Reset a forgotten password using the recovery link."),
    ("assignment",
     "This synthetic assessment section explains how coursework is "
     "delivered. Upload assignments through the learning portal. Submit the "
     "final file as a PDF. Files in other formats are not opened by the "
     "markers, and a document that cannot be opened is treated as not "
     "submitted. The portal accepts files up to twenty megabytes, so large "
     "figures should be compressed before the upload, and the student must "
     "keep a copy of the confirmation message that appears after every "
     "successful upload because the confirmation is the only proof of "
     "submission that the office accepts during a dispute. Group projects "
     "are uploaded by the team leader only, and every team member must be "
     "named on the first page of the document. Laboratory reports for "
     "practical sessions are different because they are handed to the "
     "instructor on paper at the start of the next session and are not "
     "accepted through the portal. Handwritten work must be scanned clearly "
     "before it is uploaded, and photographs taken at an angle may be "
     "rejected by the coordinator. Submissions are accepted until midnight "
     "on the deadline day according to the portal clock, and a "
     "resubmission before the deadline replaces the earlier file without "
     "any penalty.",
     "Upload assignments through the learning portal."),
    ("contractor",
     "This synthetic workplace policy describes remote work rules for "
     "different groups of people. Employees may work remotely for up to two "
     "days each week when the line manager approves the plan in advance. "
     "Contractors can work remotely only if the signed contract permits it. "
     "Employee rules do not apply to contractors. A contractor whose "
     "contract is silent on remote work must work on site until the "
     "contract is amended by the human resources office. Visitors and "
     "interns follow a separate internship policy and should not use this "
     "section. Remote work from another country is not allowed for any "
     "group without written approval from the legal office, because tax and "
     "data protection rules differ between countries. Everyone who works "
     "remotely must use the company virtual private network and must keep "
     "work devices locked when they are not in use. Managers review remote "
     "work plans every quarter, and a plan can be withdrawn with one week "
     "of notice when the needs of the team change. Equipment such as "
     "laptops remains the property of the company and must be returned at "
     "the end of the arrangement.",
     "Contractors can work remotely only if the signed contract permits it."),
    ("printer",
     "This synthetic equipment guide collects short troubleshooting notes "
     "for office devices. For blank printer pages, check ink levels and run "
     "the cleaning utility. Print a test page afterwards. If the test page "
     "is still blank, open the printer menu and make sure that the correct "
     "paper size is selected, because a wrong size can cause empty output. "
     "Warning: switch the device off and wait ten minutes before touching "
     "any internal part, since the heating unit stays hot. For a blank "
     "scanner image, check document placement and select the correct "
     "scanning source. A scanner glass with fingerprints can also lower "
     "quality, so clean it with a soft dry cloth. A printer that makes "
     "noise but pulls no paper usually has a dirty roller, and the roller "
     "can be wiped with a slightly damp cloth after unplugging the cable. "
     "Streaks across the page point to a clogged print head, and a second "
     "cleaning cycle is usually enough. Each device model has its own "
     "manual, and the instructions for one model must not be applied to "
     "another model without checking. Replace cartridges only with the type "
     "listed in the manual.",
     "For blank printer pages, check ink levels and run the cleaning utility."),
]

corpus = {name: body for name, body, _ in entries}
gold = {name: ev for name, _, ev in entries}

qa_pairs = [
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("What are tomorrow's flight prices?", None, "test"),
    ("Who won the football match?", None, "test"),
]

labelled = [
    {"query": q, "doc": d, "evidence": gold[d] if d else None, "split": s}
    for q, d, s in qa_pairs
]

overwrite = "--force" in sys.argv
for filename, payload in (("corpus.json", corpus), ("queries.json", labelled)):
    target = Path(filename)
    if target.exists() and not overwrite:
        raise FileExistsError("Refusing to overwrite " + filename + " (use --force)")
    target.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

for name, body in corpus.items():
    assert 150 <= len(body.split()) <= 250, name
    assert gold[name] in body
print("Created corpus.json and queries.json")
print({name: len(body.split()) for name, body in corpus.items()}, "words per document")

Writing create_dataset.py


In [5]:
!python create_dataset.py --force

Created corpus.json and queries.json
{'late': 198, 'password': 182, 'assignment': 196, 'contractor': 183, 'printer': 187} words per document


In [6]:
%%writefile guided_metrics.py
def retrieval_metrics(rankings, relevant):
    found, recip = [], []
    for ordered, wanted in zip(rankings, relevant):
        positions = [pos for pos, cid in enumerate(ordered[:3], 1) if cid in wanted]
        found.append(bool(positions))
        recip.append(1 / positions[0] if positions else 0)
    return sum(found) / len(found), sum(recip) / len(recip)


hit, mrr = retrieval_metrics(
    [["a", "b", "c"], ["c", "a", "b"], ["x", "y", "z"]],
    [{"a"}, {"a"}, {"a"}])
print(f"Hit@3={hit:.3f}; MRR@3={mrr:.3f}")
assert abs(hit - 2 / 3) < 1e-9 and abs(mrr - 0.5) < 1e-9

h4, m4 = retrieval_metrics([["x", "y", "z", "a"]], [{"a"}])
print(f"Rank-4 case: Hit@3={h4:.3f}; MRR@3={m4:.3f}")
assert h4 == 0 and m4 == 0

Writing guided_metrics.py


In [7]:
!python guided_metrics.py

Hit@3=0.667; MRR@3=0.500
Rank-4 case: Hit@3=0.000; MRR@3=0.000


In [8]:
%%writefile task3_retrieval.py
import csv
import json
import time

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

from task2_chunks import make_chunks

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def is_relevant(chunk, item):
    return (item["doc"] is not None and chunk["doc"] == item["doc"]
            and item["evidence"] in chunk["text"])


def evaluate(scores, chunks, items):
    flags, recips = [], []
    for row, item in zip(scores, items):
        if item["doc"] is None:
            continue
        top3 = np.argsort(-row, kind="stable")[:3]
        places = [p for p, idx in enumerate(top3, 1) if is_relevant(chunks[idx], item)]
        flags.append(bool(places))
        recips.append(1 / places[0] if places else 0)
    if not flags:
        return None, None
    return float(np.mean(flags)), float(np.mean(recips))


def load_inputs():
    with open("corpus.json", encoding="utf-8") as fh:
        corpus = json.load(fh)
    with open("queries.json", encoding="utf-8") as fh:
        items = json.load(fh)
    assert corpus and items
    for item in items:
        assert item["split"] in {"dev", "test"}
        if item["doc"] is not None:
            assert item["evidence"] and item["evidence"] in corpus[item["doc"]]
    return corpus, items


def main():
    corpus, items = load_inputs()

    started = time.perf_counter()
    encoder = SentenceTransformer(MODEL_NAME, device="cpu")
    load_seconds = time.perf_counter() - started
    print("Model load seconds", load_seconds)

    queries = [item["query"] for item in items]
    log_rows, summary = [], []

    for strategy in ("0", "10", "sentence"):
        chunks = make_chunks(corpus, strategy)
        texts = [ch["text"] for ch in chunks]
        sizes = [len(encoder.tokenizer(t, truncation=False)["input_ids"])
                 for t in texts + queries]
        print(f"[audit] strategy={strategy} max tokens={max(sizes)} "
              f"limit={encoder.max_seq_length}")
        if max(sizes) > encoder.max_seq_length:
            raise ValueError("Model limit exceeded; rechunk or shorten query")

        for method in ("tfidf", "embedding"):
            began = time.perf_counter()
            if method == "tfidf":
                vectorizer = TfidfVectorizer()
                doc_side = vectorizer.fit_transform(texts)
                query_side = vectorizer.transform(queries)
            else:
                doc_side = encoder.encode(texts, normalize_embeddings=True)
                query_side = encoder.encode(queries, normalize_embeddings=True)
            encode_seconds = time.perf_counter() - began

            began = time.perf_counter()
            product = query_side @ doc_side.T
            scores = product.toarray() if method == "tfidf" else product
            search_seconds = time.perf_counter() - began

            for split in ("dev", "test"):
                chosen = [i for i, item in enumerate(items) if item["split"] == split]
                hit, mrr = evaluate(scores[chosen], chunks, [items[i] for i in chosen])
                summary.append((strategy, method, split, hit, mrr,
                                len(chunks), encode_seconds, search_seconds))
                print(strategy, method, split, "Hit@3/MRR@3", (hit, mrr),
                      "chunks", len(chunks), "encode/search seconds",
                      round(encode_seconds, 4), round(search_seconds, 6))

            for row, item in zip(scores, items):
                top3 = np.argsort(-row, kind="stable")[:3]
                any_relevant = False
                for place, idx in enumerate(top3, 1):
                    ch = chunks[idx]
                    good = is_relevant(ch, item)
                    any_relevant = any_relevant or good
                    log_rows.append({
                        "strategy": strategy, "method": method,
                        "query": item["query"], "split": item["split"],
                        "rank": place, "score": float(row[idx]),
                        "chunk_id": ch["id"], "doc": ch["doc"],
                        "start": ch["start"], "end": ch["end"],
                        "text": ch["text"], "answerable": item["doc"] is not None,
                        "relevant": good})
                lead = chunks[top3[0]]
                if item["doc"] is not None and not any_relevant:
                    print(f"   FAILURE [{strategy}/{method}] {item['query']!r} -> "
                          f"top1 {lead['id']} (score {row[top3[0]]:.3f})")
                if item["doc"] is None:
                    print(f"   UNANSWERABLE [{strategy}/{method}] {item['query']!r} -> "
                          f"top1 {lead['id']} score {row[top3[0]]:.3f}")

    dev_rows = [s for s in summary if s[2] == "dev"]
    best = max(dev_rows, key=lambda s: (s[4], s[3], -s[5]))
    print("\nBest on DEV (by MRR@3, then Hit@3, then fewer chunks):",
          best[0], best[1], "Hit/MRR", best[3], best[4])
    for s in summary:
        if s[2] == "test" and s[0] == best[0] and s[1] == best[1]:
            print("Held-out TEST result for that config (no retuning):",
                  "Hit/MRR", s[3], s[4])
    print("Model load seconds (reported separately):", load_seconds)

    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(log_rows[0]))
        writer.writeheader()
        writer.writerows(log_rows)
    print("Wrote retrieval_log.csv with", len(log_rows), "rows")


if __name__ == "__main__":
    main()

Writing task3_retrieval.py


In [9]:
!python task3_retrieval.py

modules.json: 100% 349/349 [00:00<00:00, 913kB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 401kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 21.6MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 225kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.84MB/s]

model.safetensors: downloading bytes:  81% 73.2M/90.9M [00:01<00:00, 93.5MB/s, 5.68MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 57.2MB/s, 8.11MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 61.2MB/s, 8.80MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6054.68it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.44MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 41.3MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 80.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 494kB/s]
config.json: 100% 190/190 [00:00<00:00, 752kB/s]
Model load seconds 5.280388948999985
[audit] strategy=0 max tokens=51 limit=256
0 tfidf dev Hit@3/MRR@3 

In [10]:
import pandas as pd
df = pd.read_csv('retrieval_log.csv')
print(df.shape)
df[['strategy', 'method', 'query', 'split', 'rank', 'score', 'chunk_id', 'start', 'end', 'relevant']].head(12)

(180, 13)


,strategy,method,query,split,rank,score,chunk_id,start,end,relevant
0,0,tfidf,What is required for illness extensions?,dev,1,0.235149,late:2,512,762,False
1,0,tfidf,What is required for illness extensions?,dev,2,0.234137,late:1,247,511,False
2,0,tfidf,What is required for illness extensions?,dev,3,0.139498,late:0,0,246,False
3,0,tfidf,How do I reset a forgotten password?,dev,1,0.230966,password:0,0,235,True
4,0,tfidf,How do I reset a forgotten password?,dev,2,0.083603,contractor:1,243,485,False
5,0,tfidf,How do I reset a forgotten password?,dev,3,0.075425,password:2,458,668,False
6,0,tfidf,Where do I upload assignments?,dev,1,0.191787,assignment:1,246,493,False
7,0,tfidf,Where do I upload assignments?,dev,2,0.191700,assignment:0,0,245,True
8,0,tfidf,Where do I upload assignments?,dev,3,0.107109,contractor:1,243,485,False
9,0,tfidf,Can contractors work remotely?,dev,1,0.475793,contractor:0,0,242,False


In [ ]:
df[(~df.answerable) & (df['rank'] == 1)][['strategy', 'method', 'query', 'chunk_id', 'score']]

In [11]:
answerable = df[df.answerable]
found = answerable.groupby(['strategy', 'method', 'query']).relevant.any().reset_index()
found[~found.relevant]

,strategy,method,query,relevant
0,0,embedding,Can contractors work remotely?,False
5,0,embedding,What is required for illness extensions?,False
6,0,embedding,What proof is needed when I was ill?,False
8,0,tfidf,Can contractors work remotely?,False
13,0,tfidf,What is required for illness extensions?,False
14,0,tfidf,What proof is needed when I was ill?,False
30,10,tfidf,What proof is needed when I was ill?,False
38,sentence,embedding,What proof is needed when I was ill?,False
46,sentence,tfidf,What proof is needed when I was ill?,False


In [12]:
%%writefile document_chatbot.py
import argparse
import json

from task2_chunks import make_chunks
from lab6_search import Retriever

STRATEGY = "sentence"


def run(backend):
    with open("corpus.json", encoding="utf-8") as fh:
        corpus = json.load(fh)
    engine = Retriever(make_chunks(corpus, STRATEGY), backend)
    print("Evidence assistant. Type exit to stop.")
    while True:
        try:
            question = input("Question: ")
        except (EOFError, KeyboardInterrupt):
            break
        if question.strip().lower() == "exit":
            break
        candidates = engine.search(question)
        if not candidates:
            print("No candidate evidence; rephrase or contact a human.")
            continue
        print("Candidate evidence; relevance must be checked:")
        for number, (rec, score) in enumerate(candidates, 1):
            print(number, rec["id"], f"{score:.3f}", "characters", rec["start"], rec["end"])
        try:
            pick = input("Choose 1-3 to inspect; Enter if none apply: ")
        except (EOFError, KeyboardInterrupt):
            break
        if not pick.isdigit() or not 1 <= int(pick) <= len(candidates):
            print("No answer selected. Rephrase or contact a human.")
            continue
        rec, score = candidates[int(pick) - 1]
        print("Retrieved passage:", rec["text"])
        print("Source:", rec["doc"], "characters", rec["start"], rec["end"])


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--backend", choices=["tfidf", "embedding"], default="embedding")
    run(parser.parse_args().backend)

Writing document_chatbot.py


In [13]:
import json
from task2_chunks import make_chunks
from lab6_search import Retriever
import document_chatbot as dc
corpus = json.load(open('corpus.json', encoding='utf-8'))
engine = Retriever(make_chunks(corpus, dc.STRATEGY), 'embedding')
for q in ['How can I recover my login access?', '', 'Who won the football match?']:
    print('Question:', repr(q))
    hits = engine.search(q)
    if not hits:
        print('No candidate evidence; rephrase or contact a human.')
        print()
        continue
    print('Candidate evidence; relevance must be checked:')
    for n, (rec, s) in enumerate(hits, 1):
        print(n, rec['id'], f'{s:.3f}', 'characters', rec['start'], rec['end'])
    rec, s = hits[0]
    print('Retrieved passage (choice 1):', rec['text'])
    print('Source:', rec['doc'], 'characters', rec['start'], rec['end'])
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Question: 'How can I recover my login access?'
Candidate evidence; relevance must be checked:
1 password:0 0.648 characters 0 131
2 password:4 0.426 characters 595 805
3 password:1 0.408 characters 131 352
Retrieved passage (choice 1): This synthetic IT help section explains account access for students and staff. Reset a forgotten password using the recovery link. 
Source: password characters 0 131

Question: ''
No candidate evidence; rephrase or contact a human.

Question: 'Who won the football match?'
Candidate evidence; relevance must be checked:
1 assignment:3 0.122 characters 519 580
2 password:1 0.116 characters 131 352
3 printer:2 0.086 characters 345 558
Retrieved passage (choice 1): proof of submission that the office accepts during a dispute.
Source: assignment characters 519 580



In [14]:
%run document_chatbot.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evidence assistant. Type exit to stop.
Question: exit


In [15]:
from google.colab import files
for name in ['retrieval_log.csv', 'corpus.json', 'queries.json']:
    files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('Task3', 'zip', '.')
files.download('Task3.zip')